# RUSH 4 — Segmentation Instagram × bien-être
## 03 · Approche hybride : intensité d'usage × style de contenu

**Question.** Peut-on construire **4 à 7 segments** qui (a) décrivent une façon d'utiliser Instagram,
(b) sont liés au bien-être, (c) sont assignables à un nouvel utilisateur et (d) font mieux que le
simple découpage par temps passé ?

**Réponse courte.** L'**intensité d'usage** porte l'essentiel du lien au bien-être. Le **style de
contenu** (Reels vs Feed) est un axe descriptif utile pour le produit, mais il n'ajoute **aucune**
variance de bien-être dans ces données. Ce notebook le démontre, chiffres à l'appui, et en tire une
recommandation prudente : 6 segments pour la stratégie produit, 3 bandes de temps pour le risque
bien-être, et **aucun ciblage santé ou mineur**.

### Pourquoi une troisième route ?

| Notebook | Point de départ | Méthode | Ce que l'on regarde |
|---|---|---|---|
| `01_audit_exploration.ipynb` | Variables d'usage | KMeans + seuils de temps | Existe-t-il des clusters naturels ? |
| `02_approche_supervisee.ipynb` | Indicateur de bien-être | Modèle supervisé → probabilité → 3 niveaux de risque | Peut-on prédire le stress ? |
| **`03_notebook_final_hybride.ipynb`** | **Temps × style** | **Grille 3 × 2, tests de critères A/B/C** | **Le style ajoute-t-il quelque chose au temps ?** |

Cette troisième route suit explicitement le cahier des charges « Rush 4 – Livrables et tâches »
(binômes A/B/C, T0/T1, A1–A5, B1–B5, C1–C6). Elle est volontairement **honnête** : elle montre
aussi ce qui ne marche pas.

> **Règle de confidentialité.** Aucune ligne individuelle n'est affichée. Tous les tableaux sont
> des agrégats. Les profils utilisés pour tester l'affectation sont **fictifs**.

### Carte des tâches

| Tâche | Où elle est traitée | Statut |
|---|---|---|
| T0 — dépôt Git, rôles, README | Section 0 | Procédure + tableau des binômes |
| T1 — classer les 58 colonnes | Section 1 | Tableau complet des 3 tiroirs |
| A1 — comparer les deux fichiers | Section 2 | 6 colonnes divergentes chiffrées |
| A2 — enquêter sur chaque différence | Section 3 | Tableau colonne / preuve / version gardée |
| A3 — jeu de données final | Section 4 | `charger_donnees()` |
| A4 — explorer usage et bien-être | Section 5 | 4 graphiques commentés |
| A5 — reproductibilité | Section 6 | Double exécution + hash |
| C1 — critères | Section 7 | 6 critères fixés avant les segments |
| B1 — référence temps | Section 8 | Tableau de référence |
| B2 — nouvelles variables | Section 9 | Ratios + justification chiffrée |
| B3 — choix de méthode | Section 10 | KMeans vs prédictif vs hybride |
| B4 — segments nommés | Section 11 | 6 fiches segments |
| B5 — affectation | Section 12 | Fonction + profils fictifs |
| C2 — tests des critères | Section 13 | 6 fonctions, verdicts |
| C3 — évaluation | Section 14 | Tableau de synthèse + renvoi à B |
| C4 — limites | Section 15 | 3 paragraphes illustrés |
| C5 — RGPD | Section 16 | UE / Royaume-Uni / mineurs |
| C6 — slides | Section 17 | Trame de 12 slides |


## 0. T0 — Dépôt Git, rôles et premier commit

Le dépôt doit montrer **qui a fait quoi dès le jour 1**. Cette section documente la procédure ;
elle ne l'exécute pas à la place de l'équipe.

| Binôme | Membres | Mission | Ce que ce notebook livre |
|---|---|---|---|
| **A — Données** | Sarah & Sophie | Choisir les données fiables, explorer, tenir le notebook propre | Sections 2 à 6 |
| **B — Segmentation** | Léonie & Emmy | Construire les variables, les segments et la règle d'affectation | Sections 8 à 12 |
| **C — Évaluation & conformité** | Stève & Myriam | Critères, preuves, limites, RGPD, slides | Sections 7, 13 à 17 |

**Checklist T0 (à faire dans le dépôt Git) :**

```bash
git init
git add instagram_usage_lifestyleSafe.csv instagram_users_lifestyleSafe.csv README_analyse.md
git commit -m "T0: depot initial, CSV et README"
# chacun des 6 membres ajoute une ligne dans le README puis commit à son nom
```

- [ ] dépôt créé et 6 membres invités ;
- [ ] les 2 CSV sont déposés ;
- [ ] un notebook vide est déposé ;
- [ ] le README contient le tableau des binômes ;
- [ ] chaque membre a au moins un commit à son nom.


## 1. Environnement et helpers

On fixe une graine unique (`RANDOM_STATE = 42`) et on évite toute dépendance à un kernel exotique.
Le notebook doit tourner tel quel dans VS Code avec `pandas`, `numpy`, `matplotlib` et `scikit-learn`.


In [ ]:
from pathlib import Path
import json
import time
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from itertools import combinations
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.pipeline import make_pipeline
from sklearn.metrics import (
    silhouette_score, adjusted_rand_score, cohen_kappa_score,
    roc_auc_score, average_precision_score, brier_score_loss,
)

try:
    from IPython.display import display
except ImportError:
    display = print

RANDOM_STATE = 42
DATA_DIR = Path(".")
FIG_DIR = Path("figures")
FIG_DIR.mkdir(exist_ok=True)

pd.set_option("display.width", 180)
pd.set_option("display.max_columns", 70)
pd.set_option("display.max_rows", 90)
pd.set_option("display.float_format", lambda x: f"{x:,.3f}")

plt.rcParams.update({
    "figure.figsize": (11, 4),
    "axes.grid": True,
    "grid.alpha": 0.3,
    "axes.spines.top": False,
    "axes.spines.right": False,
})


def save_fig(name: str, dpi: int = 120) -> None:
    """Sauvegarde robuste (OneDrive peut verrouiller un fichier quelques secondes)."""
    path = (FIG_DIR / name).resolve()
    for _ in range(3):
        try:
            plt.savefig(path, dpi=dpi)
            return
        except OSError:
            time.sleep(0.5)
    plt.savefig(path, dpi=dpi)


def eta_squared(frame: pd.DataFrame, categorical: str, target: str) -> float:
    """Part de variance de `target` expliquée par les modalités de `categorical`."""
    grand = frame[target].mean()
    grouped = frame.groupby(categorical, observed=True)[target]
    ss_between = ((grouped.mean() - grand) ** 2 * grouped.size()).sum()
    ss_total = ((frame[target] - grand) ** 2).sum()
    return float(ss_between / ss_total)


print("Environnement prêt.")


## 2. Chargement et audit des deux extraits

Les deux fichiers ont les mêmes 200 000 `user_id` et 58 colonnes, mais ils ne disent pas la même
chose sur 6 champs. On les charge **sans les concaténer** : chaque champ sera choisi séparément.


In [ ]:
path_a = DATA_DIR / "instagram_usage_lifestyleSafe.csv"
path_b = DATA_DIR / "instagram_users_lifestyleSafe.csv"

raw_a = pd.read_csv(path_a)
raw_b = pd.read_csv(path_b)

audit = pd.DataFrame({
    "Extrait A (usage)": [raw_a.shape[0], raw_a.shape[1], raw_a["user_id"].nunique(),
                          int(raw_a.duplicated().sum()), int(raw_a.isna().sum().sum())],
    "Extrait B (users)": [raw_b.shape[0], raw_b.shape[1], raw_b["user_id"].nunique(),
                          int(raw_b.duplicated().sum()), int(raw_b.isna().sum().sum())],
}, index=["Lignes", "Colonnes", "user_id uniques", "Lignes dupliquées", "Valeurs manquantes"])

display(audit)
print("Colonnes identiques :", list(raw_a.columns) == list(raw_b.columns))
print("Mêmes user_id, même ordre :", bool((raw_a["user_id"].values == raw_b["user_id"].values).all()))
print("Aucune valeur manquante :", raw_a.isna().sum().sum() == 0 and raw_b.isna().sum().sum() == 0)


## 3. T1 — Classer les 58 colonnes en trois tiroirs

**Règle de classement :**

- **Usage Instagram** : ce que la personne *fait dans l'application* (temps, sessions, clics,
  contenus consommés, audience, engagement). Ce sont les seules variables qui ont le droit de
  servir à **ranger** un utilisateur.
- **Profil** : qui elle est, hors usage (âge, genre, pays, statut, éducation, compte, sécurité).
  Utile pour décrire et contrôler l'équité, **pas** pour construire les segments.
- **Bien-être / santé** : stress, bonheur, sommeil, santé, mode de vie. Ce sont les **cibles**
  d'évaluation, jamais des entrées de segmentation.
- **Identifiant / technique** : `user_id`, `app_name`. Exclus.

> **Validation T1.** Le tableau ci-dessous est la version de travail à valider par les 6 en début
> de rush. Les cas douteux sont tranchés explicitement juste après.

### Cas douteux tranchés

| Colonne | Tiroir | Décision | Justification |
|---|---|---|---|
| `income_level` | Profil | Non utilisé | Situation socio-économique, pas un comportement d'usage ; pourrait servir de proxy sensible. |
| `relationship_status` | Profil | Non utilisé | Donnée familiale sensible, sans lien démontré avec l'usage. |
| `has_children` | Profil | Non utilisé | Donnée familiale sensible ; risque de ciblage des parents. |
| `weekly_work_hours` | Profil | Non utilisé | Temps de travail, pas un usage Instagram ; incohérent dans A pour les mineurs. |
| `daily_steps_count` | Bien-être / santé | Non utilisé | Comportement santé, pas un usage Instagram. |
| `content_type_preference` | Usage | Descriptif | Préférence déclarée ; peut décrire le style, mais lien au bien-être à tester. |
| `user_engagement_score` | Usage | Diagnostic | Score dérivé, redondant avec le temps ; ne pas utiliser comme feature principale. |


In [ ]:
COLUMN_DRAWERS = {
    "Usage Instagram": [
        "daily_active_minutes_instagram", "sessions_per_day", "posts_created_per_week",
        "reels_watched_per_day", "stories_viewed_per_day", "likes_given_per_day",
        "comments_written_per_day", "dms_sent_per_week", "dms_received_per_week",
        "ads_viewed_per_day", "ads_clicked_per_day", "time_on_feed_per_day",
        "time_on_explore_per_day", "time_on_messages_per_day", "time_on_reels_per_day",
        "followers_count", "following_count", "notification_response_rate",
        "average_session_length_minutes", "content_type_preference",
        "preferred_content_theme", "user_engagement_score",
    ],
    "Profil": [
        "age", "gender", "country", "urban_rural", "income_level", "employment_status",
        "education_level", "relationship_status", "has_children", "weekly_work_hours",
        "uses_premium_features", "account_creation_year", "last_login_date",
        "privacy_setting_level", "two_factor_auth_enabled", "biometric_login_used",
        "linked_accounts_count", "subscription_status",
    ],
    "Bien-être / santé": [
        "exercise_hours_per_week", "sleep_hours_per_night", "diet_quality", "smoking",
        "alcohol_frequency", "perceived_stress_score", "self_reported_happiness",
        "body_mass_index", "blood_pressure_systolic", "blood_pressure_diastolic",
        "daily_steps_count", "hobbies_count", "social_events_per_month",
        "books_read_per_year", "volunteer_hours_per_month", "travel_frequency_per_year",
    ],
    "Identifiant / technique": ["user_id", "app_name"],
}

SUBCATS = {
    "daily_active_minutes_instagram": "Temps & intensité",
    "sessions_per_day": "Temps & intensité",
    "average_session_length_minutes": "Temps & intensité",
    "time_on_feed_per_day": "Surface de contenu",
    "time_on_reels_per_day": "Surface de contenu",
    "time_on_messages_per_day": "Surface de contenu",
    "time_on_explore_per_day": "Surface de contenu",
    "content_type_preference": "Préférence déclarée",
    "preferred_content_theme": "Préférence déclarée",
    "user_engagement_score": "Score dérivé",
    "posts_created_per_week": "Création",
    "reels_watched_per_day": "Consommation",
    "stories_viewed_per_day": "Consommation",
    "likes_given_per_day": "Interaction",
    "comments_written_per_day": "Interaction",
    "dms_sent_per_week": "Interaction",
    "dms_received_per_week": "Interaction",
    "ads_viewed_per_day": "Publicité",
    "ads_clicked_per_day": "Publicité",
    "followers_count": "Audience",
    "following_count": "Audience",
    "notification_response_rate": "Réactivité",
    "age": "Socio-démographie", "gender": "Socio-démographie", "country": "Socio-démographie",
    "urban_rural": "Socio-démographie", "income_level": "Socio-démographie",
    "employment_status": "Socio-démographie", "education_level": "Socio-démographie",
    "relationship_status": "Socio-démographie", "has_children": "Socio-démographie",
    "weekly_work_hours": "Travail",
    "account_creation_year": "Compte", "last_login_date": "Compte",
    "privacy_setting_level": "Sécurité", "two_factor_auth_enabled": "Sécurité",
    "biometric_login_used": "Sécurité", "linked_accounts_count": "Compte",
    "subscription_status": "Produit", "uses_premium_features": "Produit",
    "perceived_stress_score": "Santé mentale", "self_reported_happiness": "Bien-être",
    "sleep_hours_per_night": "Santé", "body_mass_index": "Santé",
    "blood_pressure_systolic": "Santé", "blood_pressure_diastolic": "Santé",
    "exercise_hours_per_week": "Mode de vie", "diet_quality": "Mode de vie",
    "smoking": "Mode de vie", "alcohol_frequency": "Mode de vie",
    "daily_steps_count": "Mode de vie", "hobbies_count": "Mode de vie",
    "social_events_per_month": "Mode de vie", "books_read_per_year": "Mode de vie",
    "volunteer_hours_per_month": "Mode de vie", "travel_frequency_per_year": "Mode de vie",
}

USAGE_FLAGS = {
    "daily_active_minutes_instagram": "Oui — axe intensité",
    "time_on_reels_per_day": "Oui — axe style",
    "time_on_feed_per_day": "Oui — axe style",
    "time_on_messages_per_day": "Descriptif",
    "time_on_explore_per_day": "Descriptif",
    "sessions_per_day": "Descriptif",
    "average_session_length_minutes": "Descriptif",
    "reels_watched_per_day": "Descriptif",
    "stories_viewed_per_day": "Descriptif",
    "likes_given_per_day": "Descriptif",
    "comments_written_per_day": "Descriptif",
    "dms_sent_per_week": "Descriptif",
    "dms_received_per_week": "Descriptif",
    "ads_viewed_per_day": "Descriptif",
    "ads_clicked_per_day": "Descriptif",
    "posts_created_per_week": "Descriptif",
    "followers_count": "Descriptif",
    "following_count": "Descriptif",
    "notification_response_rate": "Descriptif",
    "content_type_preference": "Descriptif",
    "preferred_content_theme": "Descriptif",
    "user_engagement_score": "Diagnostic, non retenu",
}

rows = []
for drawer, cols in COLUMN_DRAWERS.items():
    for col in cols:
        if drawer == "Bien-être / santé":
            flag = "Non — évaluation seulement"
        elif drawer == "Profil":
            flag = "Non"
        elif drawer == "Identifiant / technique":
            flag = "Non"
        else:
            flag = USAGE_FLAGS.get(col, "Descriptif")
        rows.append({
            "Colonne": col,
            "Tiroir": drawer,
            "Sous-catégorie": SUBCATS.get(col, drawer),
            "Usage dans la segmentation": flag,
        })

drawer_table = pd.DataFrame(rows)
all_classified = [c for cols in COLUMN_DRAWERS.values() for c in cols]

assert len(all_classified) == 58, len(all_classified)
assert len(set(all_classified)) == 58
assert set(all_classified) == set(raw_a.columns), set(raw_a.columns) - set(all_classified)

display(drawer_table.groupby(["Tiroir", "Usage dans la segmentation"]).size()
        .rename("Nombre de colonnes").to_frame())
display(drawer_table)
print("Les 58 colonnes sont classées une et une seule fois.")


## 4. A1 — Comparer les deux fichiers colonne par colonne

On aligne les deux extraits sur `user_id` (même ordre, contrôle explicite) et on compte les
différences. Le cahier des charges annonce **6 colonnes** divergentes ; on le vérifie.


In [ ]:
def divergence_profile(a: pd.DataFrame, b: pd.DataFrame) -> pd.DataFrame:
    """Compte les lignes qui diffèrent, colonne par colonne."""
    rows = []
    for col in a.columns:
        if pd.api.types.is_numeric_dtype(a[col]):
            diff = ~np.isclose(a[col].astype(float), b[col].astype(float), equal_nan=True)
            kind = "numérique"
        else:
            diff = a[col].fillna("NA").astype(str) != b[col].fillna("NA").astype(str)
            kind = "catégorielle"
        n = int(diff.sum())
        if n:
            rows.append({"Colonne": col, "Type": kind, "Lignes divergentes": n,
                         "% divergentes": round(100 * n / len(a), 2)})
    return pd.DataFrame(rows).sort_values("Lignes divergentes", ascending=False)


raw_a_sorted = raw_a.sort_values("user_id").reset_index(drop=True)
raw_b_sorted = raw_b.sort_values("user_id").reset_index(drop=True)
assert (raw_a_sorted["user_id"].values == raw_b_sorted["user_id"].values).all()
print("Fichiers triés par user_id et alignés :", len(raw_a_sorted), "lignes.")

divergences = divergence_profile(raw_a_sorted, raw_b_sorted)
display(divergences)
print("Nombre de colonnes divergentes :", len(divergences))
print("Colonnes :", list(divergences["Colonne"]))
assert len(divergences) == 6

# Contrôles internes : les incohérences que le cahier des charges demande d'investiguer.
login_a = pd.to_datetime(raw_a["last_login_date"], errors="coerce").dt.year
login_b = pd.to_datetime(raw_b["last_login_date"], errors="coerce").dt.year
birth_a = login_a - raw_a["age"]
birth_b = login_b - raw_b["age"]

checks = []
for name, frame, login, birth in [("A", raw_a, login_a, birth_a), ("B", raw_b, login_b, birth_b)]:
    minors = frame["age"] < 18
    checks.append({
        "Extrait": name,
        "Mineurs": int(minors.sum()),
        "Mineurs mariés/divorcés/veufs": int((minors & frame["relationship_status"].isin(
            ["Married", "Divorced", "Widowed"])).sum()),
        "Mineurs avec enfants": int((minors & (frame["has_children"] == "Yes")).sum()),
        "Mineurs avec doctorat": int((minors & (frame["education_level"] == "PhD")).sum()),
        "Mineurs > 20 h de travail": int((minors & (frame["weekly_work_hours"] > 20)).sum()),
        "Création < 13 ans": int((frame["account_creation_year"] < birth + 13).sum()),
        "Création > dernière connexion": int((frame["account_creation_year"] > login).sum()),
        "Création en 2026": int((frame["account_creation_year"] == 2026).sum()),
    })

display(pd.DataFrame(checks).set_index("Extrait"))
print("Heures de travail moyennes des mineurs — A :",
      round(raw_a.loc[raw_a["age"] < 18, "weekly_work_hours"].mean(), 1),
      "| B :", round(raw_b.loc[raw_b["age"] < 18, "weekly_work_hours"].mean(), 1))


## 5. A2 — Enquêter sur chaque différence

Aucun fichier n'est parfait. Le raisonnement est **champ par champ** : quel extrait est le plus
cohérent, et que sait-on des utilisateurs concernés ?

### Tableau de décision

| Colonne | Version gardée | Preuve | Conséquence |
|---|---|---|---|
| `employment_status` | **B** | A donne un statut actif (full-time, part-time, retraité…) à des mineurs ; B met les 18 913 mineurs en `Not employed`. | B est cohérent avec l'âge. |
| `weekly_work_hours` | **B** | A : 18 464 mineurs à plus de 20 h/semaine, moyenne 40 h ; B : 0 h pour tous les mineurs. | B est cohérent avec le statut. |
| `relationship_status` | **B** | A : 6 623 mineurs mariés, divorcés ou veufs ; B : seulement `Single` ou `In a relationship`. | B protège les mineurs. |
| `has_children` | **B** | A : 5 329 mineurs avec enfants ; B : 0. | B est cohérent. |
| `education_level` | **B, avec réserve** | A écrase 1 203 libellés en `PhD`, dont 575 mineurs ; B restaure `Secondary`, `Bachelor`, `High School`. Mais B laisse 8 828 mineurs en Bachelor's/Master's. | Meilleure que A, mais encore imparfaite ; **non utilisée** dans les segments. |
| `account_creation_year` | **A, corrigé** | A : 27 989 comptes créés avant 13 ans (81,0 % des mineurs), 0 date future ; B : 3 724 comptes créés après la dernière connexion, 3 768 en 2026, 0 avant 13 ans. | A est la base ; on borne la date entre 13 ans et la dernière connexion. |

**Lecture.** B ressemble à une passe de nettoyage ciblée sur les mineurs : elle corrige les
combinaisons socio-démographiques impossibles, mais elle introduit des dates de création futures.
A conserve des dates plausibles mais des profils de mineurs impossibles. On prend donc **le meilleur
de chaque fichier**, et on documente les limites qui restent.


In [ ]:
n_minors = int((raw_a["age"] < 18).sum())
a_minor_married = int(((raw_a["age"] < 18) & raw_a["relationship_status"].isin(
    ["Married", "Divorced", "Widowed"])).sum())
a_minor_kids = int(((raw_a["age"] < 18) & (raw_a["has_children"] == "Yes")).sum())
a_minor_phd = int(((raw_a["age"] < 18) & (raw_a["education_level"] == "PhD")).sum())
a_minor_work = int(((raw_a["age"] < 18) & (raw_a["weekly_work_hours"] > 20)).sum())
a_created_young = int((raw_a["account_creation_year"] < birth_a + 13).sum())
b_created_after = int((raw_b["account_creation_year"] > login_b).sum())
b_created_2026 = int((raw_b["account_creation_year"] == 2026).sum())
b_minor_secondary = int(((raw_b["age"] < 18) & (raw_b["education_level"] == "Secondary")).sum())

decision_table = pd.DataFrame([
    {"Colonne": "employment_status", "Divergentes": int(divergences.set_index("Colonne").loc["employment_status", "Lignes divergentes"]),
     "Preuve": f"{n_minors:,} mineurs ; A les répartit en full-time/part-time/retraité, B les met tous en Not employed.",
     "Version gardée": "B"},
    {"Colonne": "weekly_work_hours", "Divergentes": int(divergences.set_index("Colonne").loc["weekly_work_hours", "Lignes divergentes"]),
     "Preuve": f"A : {a_minor_work:,} mineurs > 20 h, moyenne 40 h ; B : 0 h pour les mineurs.",
     "Version gardée": "B"},
    {"Colonne": "relationship_status", "Divergentes": int(divergences.set_index("Colonne").loc["relationship_status", "Lignes divergentes"]),
     "Preuve": f"A : {a_minor_married:,} mineurs mariés/divorcés/veufs ; B : Single ou In a relationship.",
     "Version gardée": "B"},
    {"Colonne": "has_children", "Divergentes": int(divergences.set_index("Colonne").loc["has_children", "Lignes divergentes"]),
     "Preuve": f"A : {a_minor_kids:,} mineurs avec enfants ; B : 0.",
     "Version gardée": "B"},
    {"Colonne": "education_level", "Divergentes": int(divergences.set_index("Colonne").loc["education_level", "Lignes divergentes"]),
     "Preuve": f"A écrase {a_minor_phd:,} doctorats mineurs ; B restaure {b_minor_secondary:,} Secondary, mais laisse 8 828 mineurs en Bachelor's/Master's.",
     "Version gardée": "B, avec réserve"},
    {"Colonne": "account_creation_year", "Divergentes": int(divergences.set_index("Colonne").loc["account_creation_year", "Lignes divergentes"]),
     "Preuve": f"A : {a_created_young:,} créations < 13 ans, 0 future ; B : {b_created_after:,} après dernière connexion, {b_created_2026:,} en 2026.",
     "Version gardée": "A, corrigé"},
])
display(decision_table)


## 6. A3 — Construire le jeu de données final

`charger_donnees()` est le **contrat unique** de l'équipe :

1. partir de A pour l'usage et le bien-être ;
2. remplacer les 5 champs socio-démographiques par B ;
3. harmoniser l'éducation ;
4. borner `account_creation_year` entre 13 ans et la dernière connexion ;
5. ajouter les drapeaux `is_minor`, `usage_floor_flag`, `education_minor_implausible`.

La fonction ne renvoie jamais une ligne individuelle : elle renvoie le tableau complet, et les
cellules suivantes n'affichent que des agrégats.


In [ ]:
B_SOCIO = ["employment_status", "weekly_work_hours", "relationship_status",
           "has_children", "education_level"]
EDU_MAP = {"Bachelor": "Bachelor’s", "High School": "High school", "Secondary": "High school"}
TIME_COLS = ["time_on_feed_per_day", "time_on_explore_per_day",
             "time_on_messages_per_day", "time_on_reels_per_day"]


def charger_donnees(path_a=None, path_b=None) -> pd.DataFrame:
    """Assemble le jeu final : A pour l'usage/bien-être, B pour le socio-démographique.

    Déterministe, sans affichage de lignes individuelles.
    """
    path_a = Path(path_a) if path_a is not None else DATA_DIR / "instagram_usage_lifestyleSafe.csv"
    path_b = Path(path_b) if path_b is not None else DATA_DIR / "instagram_users_lifestyleSafe.csv"

    a = pd.read_csv(path_a)
    b = pd.read_csv(path_b)

    # On retire les 5 colonnes de A pour éviter les suffixes, puis on merge B sur user_id.
    a = a.drop(columns=B_SOCIO)
    df = a.merge(b[["user_id"] + B_SOCIO], on="user_id", how="left", validate="one_to_one")

    # Éducation : on garde B, on harmonise les variantes de libellé.
    df["education_level"] = df["education_level"].replace(EDU_MAP)

    # Dates : A est la base, bornée par l'âge et la dernière connexion.
    df["last_login_date"] = pd.to_datetime(df["last_login_date"], errors="coerce")
    login_year = df["last_login_date"].dt.year
    birth_year = login_year - df["age"]
    floor = birth_year + 13
    creation = df["account_creation_year"].astype(int).clip(lower=floor, upper=login_year)
    df["account_creation_year"] = creation
    df["account_age_years"] = login_year - creation

    df["is_minor"] = (df["age"] < 18).astype(int)
    df["usage_floor_flag"] = (df["daily_active_minutes_instagram"] == 5).astype(int)
    df["education_minor_implausible"] = (
        (df["is_minor"] == 1) & df["education_level"].isin(["Bachelor’s", "Master’s"])
    ).astype(int)

    return df


df = charger_donnees()
login_year = df["last_login_date"].dt.year
birth_year = login_year - df["age"]

assert len(df) == 200_000
assert df["user_id"].is_unique
assert df.isna().sum().sum() == 0
assert set(raw_a.columns).issubset(df.columns)
assert (df.loc[df["is_minor"] == 1, "employment_status"] == "Not employed").all()
assert (df.loc[df["is_minor"] == 1, "weekly_work_hours"] == 0).all()
assert (df.loc[df["is_minor"] == 1, "has_children"] == "No").all()
assert df.loc[df["is_minor"] == 1, "relationship_status"].isin(["Single", "In a relationship"]).all()
assert (df["account_creation_year"] >= birth_year + 13).all()
assert (df["account_creation_year"] <= login_year).all()

print(f"Dimensions finales : {df.shape[0]:,} × {df.shape[1]}")
print(f"Mineurs : {int(df['is_minor'].sum()):,} ({df['is_minor'].mean():.1%})")
print(f"Éducation encore implausible pour des mineurs : {int(df['education_minor_implausible'].sum()):,}")
print(f"Utilisateurs à exactement 5,0 min/jour (plancher) : {int(df['usage_floor_flag'].sum()):,}")
print("Année de création après nettoyage :")
display(df["account_creation_year"].describe().round(2).to_frame().T)
display(df["education_level"].value_counts().rename_axis("education_level").to_frame("n"))
display(df[["age", "account_age_years", "daily_active_minutes_instagram"]].describe().T.round(2))


## 7. A4 — Explorer l'usage et le bien-être

Quatre graphiques, quatre conclusions. On cherche tôt les liens **et les liens nuls** : un
indicateur de bien-être qui ne bouge pas est un résultat à annoncer, pas un échec.


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 7))
axes[0, 0].hist(df["perceived_stress_score"], bins=40, color="#4C72B0")
axes[0, 0].set(title="Stress perçu (0–40)", xlabel="score", ylabel="nb users")
axes[0, 1].hist(df["self_reported_happiness"], bins=10, color="#55A868", rwidth=0.85)
axes[0, 1].set(title="Bonheur auto-déclaré (1–10)", xlabel="score", ylabel="nb users")
axes[1, 0].hist(df["sleep_hours_per_night"], bins=40, color="#C44E52")
axes[1, 0].set(title="Sommeil (heures/nuit)", xlabel="heures", ylabel="nb users")
axes[1, 1].hist(np.log1p(df["daily_active_minutes_instagram"]), bins=50, color="#8172B2")
axes[1, 1].set(title="log(1 + minutes actives/jour)", xlabel="log(minutes)", ylabel="nb users")
plt.tight_layout()
save_fig("03_01_distributions.png")
plt.show()

hm_cols = [
    "perceived_stress_score", "self_reported_happiness", "sleep_hours_per_night",
    "daily_active_minutes_instagram", "sessions_per_day", "average_session_length_minutes",
    "reels_watched_per_day", "stories_viewed_per_day", "likes_given_per_day",
    "dms_sent_per_week", "notification_response_rate",
]
hm = df[hm_cols].corr()
fig, ax = plt.subplots(figsize=(10, 8))
im = ax.imshow(hm.values, cmap="RdBu_r", vmin=-1, vmax=1)
ax.set_xticks(range(len(hm_cols)))
ax.set_xticklabels(hm_cols, rotation=90, fontsize=8)
ax.set_yticks(range(len(hm_cols)))
ax.set_yticklabels(hm_cols, fontsize=8)
for i in range(len(hm_cols)):
    for j in range(len(hm_cols)):
        ax.text(j, i, f"{hm.values[i, j]:.2f}", ha="center", va="center", fontsize=6)
ax.set_title("Corrélations (Pearson) — usage et bien-être")
fig.colorbar(im, ax=ax, shrink=0.8)
plt.tight_layout()
save_fig("03_02_correlation.png")
plt.show()


**Conclusion graphique 1.** Le stress est large (0–40), le bonheur est borné (1–10) et le
sommeil est très concentré autour de 7 h ; le temps d'usage est fortement asymétrique, d'où
l'échelle log.

**Conclusion graphique 2.** Le temps et les sessions sont fortement corrélés au stress ; les
variables de surface (Reels, stories, likes) le sont beaucoup moins. Le sommeil est visuellement
indépendant de tout l'usage : c'est un **résultat nul** à conserver.


In [ ]:
binned = df.copy()
binned["usage_decile"] = pd.qcut(binned["daily_active_minutes_instagram"], 10,
                                 labels=False, duplicates="drop")
deciles = binned.groupby("usage_decile").agg(
    minutes=("daily_active_minutes_instagram", "mean"),
    stress=("perceived_stress_score", "mean"),
    bonheur=("self_reported_happiness", "mean"),
    sommeil=("sleep_hours_per_night", "mean"),
)
display(deciles.round(2))

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(deciles["minutes"], deciles["stress"], marker="o", color="#C44E52", label="Stress")
ax.set(xlabel="Minutes actives / jour (moyenne du décile)", ylabel="Stress moyen (0–40)",
       title="Bien-être moyen par décile de temps d'usage")
ax2 = ax.twinx()
ax2.plot(deciles["minutes"], deciles["bonheur"], marker="s", color="#55A868", label="Bonheur")
ax2.plot(deciles["minutes"], deciles["sommeil"], marker="^", color="#4C72B0", label="Sommeil (h)")
ax2.set_ylabel("Bonheur (1–10) / Sommeil (h)")
lines = ax.get_lines() + ax2.get_lines()
ax.legend(lines, [line.get_label() for line in lines], loc="center right", fontsize=8)
plt.tight_layout()
save_fig("03_03_deciles.png")
plt.show()

categorical = ["gender", "country", "income_level", "employment_status", "education_level",
               "relationship_status", "has_children", "diet_quality", "smoking",
               "alcohol_frequency", "content_type_preference", "subscription_status"]
lifestyle = ["age", "exercise_hours_per_week", "hobbies_count", "social_events_per_month",
             "books_read_per_year", "volunteer_hours_per_month", "travel_frequency_per_year",
             "body_mass_index"]

null_rows = []
for col in categorical:
    null_rows.append({"Variable": col, "Type": "Catégorielle",
                      "Mesure": eta_squared(df, col, "perceived_stress_score")})
for col in lifestyle:
    null_rows.append({"Variable": col, "Type": "Numérique",
                      "Mesure": abs(df[col].corr(df["perceived_stress_score"]))})
null_table = pd.DataFrame(null_rows).sort_values("Mesure", ascending=False)
display(null_table.head(12).round(4))

fig, ax = plt.subplots(figsize=(9, 5))
top = null_table.head(12).iloc[::-1]
colors = ["#4C72B0" if t == "Catégorielle" else "#C44E52" for t in top["Type"]]
ax.barh(top["Variable"], top["Mesure"], color=colors)
ax.set(xlabel="η² (catégorielle) ou |r| (numérique) avec le stress",
       title="Ce qui ne bouge pas : liens faibles avec le stress")
plt.tight_layout()
save_fig("03_04_liens_nuls.png")
plt.show()


**Conclusion graphique 3.** Le stress monte et le bonheur baisse de façon monotone avec le
temps ; le sommeil reste plat. La relation est forte, mais elle n'est pas causale.

**Conclusion graphique 4.** Les variables catégorielles et de mode de vie expliquent une part
infime du stress (η² ou |r| très faibles). On ne construira pas de segments sur elles.


## 8. A5 — Reproductibilité et propreté

Deux exigences du sujet : **random_state fixé partout** et **aucune ligne individuelle affichée**.
On vérifie que deux appels à `charger_donnees()` donnent exactement le même tableau.


In [ ]:
df_check_1 = charger_donnees()
df_check_2 = charger_donnees()

assert df_check_1.equals(df_check_2)
assert df_check_1.shape == (200_000, 62)
assert df_check_1["user_id"].is_unique
assert df_check_1.isna().sum().sum() == 0

hash_1 = int(pd.util.hash_pandas_object(df_check_1, index=True).sum())
hash_2 = int(pd.util.hash_pandas_object(df_check_2, index=True).sum())
assert hash_1 == hash_2

print("Deux exécutions identiques :", df_check_1.equals(df_check_2))
print("Hash du tableau :", hash_1)
print("Aucune ligne individuelle n'est affichée : uniquement des agrégats.")
print("Colonnes de santé utilisées comme entrées de segmentation : aucune.")


## 9. C1 — Critères fixés avant de construire les segments

Ces critères sont rédigés **avant** les résultats (jour 1), pour éviter de les ajuster après coup.
Ils sont mesurables et testés en section 13.

| # | Critère | Mesure | Seuil | Pourquoi ce seuil |
|---|---|---|---|---|
| 1 | **Segments distincts** | Distance minimale entre centroïdes sur `[log_minutes, style_score]` standardisés | ≥ 0,45 | Une grille 3×2 doit occuper des cellules réellement différentes ; la silhouette classique est peu adaptée à une grille. |
| 2 | **Stables** | Kappa de Cohen entre deux moitiés de l'échantillon | ≥ 0,90 | Les seuils ne doivent pas dépendre d'un tirage. |
| 3 | **Liés au bien-être** | η² du segment sur le stress | ≥ 0,30 | Effet au moins « fort » ; le temps seul fait déjà 0,63. |
| 4 | **Plus informatifs que le temps** | Δη² (6 segments − 3 bandes de temps) | ≥ 0,01 | Justifie la complexité supplémentaire ; 1 point de variance minimum. |
| 5 | **Nombre et taille utilisables** | 4 à 7 segments ; plus petit segment ≥ 5 % | 4–7 et ≥ 5 % | Un segment minuscule n'est pas actionnable. |
| 6 | **Assignable** | 4 profils fictifs → 4 segments valides | 100 % | La règle doit fonctionner sur un nouvel utilisateur. |

> **Critère 4 :** c'est celui qui peut échouer. S'il échoue, on ne prétendra pas que la
> segmentation à 6 segments est « meilleure » que le temps ; on la présentera comme une
> segmentation **produit**, pas comme une segmentation **bien-être**.


## 10. B1 — La référence « temps passé »

C'est l'adversaire à battre : trois bandes de temps (tertiles), puis on regarde le bien-être.
Si une segmentation plus complexe ne fait pas mieux, elle doit le dire.


In [ ]:
q1, q2 = df["daily_active_minutes_instagram"].quantile([1/3, 2/3])
TIME_BINS = [0, q1, q2, np.inf]
TIME_LABELS = ["Faible", "Modéré", "Intense"]
df["time_band"] = pd.cut(df["daily_active_minutes_instagram"], bins=TIME_BINS,
                         labels=TIME_LABELS, include_lowest=True, right=True)
df["high_stress"] = (df["perceived_stress_score"] >= 30).astype(int)

baseline = df.groupby("time_band", observed=True).agg(
    n=("user_id", "size"),
    minutes=("daily_active_minutes_instagram", "mean"),
    stress=("perceived_stress_score", "mean"),
    bonheur=("self_reported_happiness", "mean"),
    sommeil=("sleep_hours_per_night", "mean"),
    stress_eleve=("high_stress", "mean"),
)
baseline["part"] = baseline["n"] / len(df)
display(baseline.round(3))

eta_time = {
    "stress": eta_squared(df, "time_band", "perceived_stress_score"),
    "bonheur": eta_squared(df, "time_band", "self_reported_happiness"),
    "sommeil": eta_squared(df, "time_band", "sleep_hours_per_night"),
}
print("Seuils des tertiles :", round(q1, 1), "min et", round(q2, 1), "min")
print("Prévalence du stress élevé (score ≥ 30/40) :", f"{df['high_stress'].mean():.1%}")
print("η² temps → stress :", round(eta_time["stress"], 3),
      "| bonheur :", round(eta_time["bonheur"], 3),
      "| sommeil :", round(eta_time["sommeil"], 3))

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].bar(baseline.index.astype(str), baseline["stress"], color="#C44E52")
axes[0].set(title="Stress moyen par bande de temps", ylabel="stress (0–40)")
axes[1].bar(baseline.index.astype(str), baseline["bonheur"], color="#55A868")
axes[1].set(title="Bonheur moyen par bande de temps", ylabel="bonheur (1–10)")
plt.tight_layout()
save_fig("03_05_baseline_temps.png")
plt.show()


**Conclusion B1.** Le temps seul explique **63 %** de la variance du stress et **10 %** du
bonheur ; le sommeil reste à **0 %**. Toute segmentation doit être comparée à ce tableau, pas
seulement à un modèle compliqué.


## 11. B2 — Créer de nouvelles variables (ratios)

Deux personnes qui passent 2 h sur l'appli peuvent l'utiliser très différemment. On construit des
ratios de **façon** d'utiliser, pas de **quantité**. Chaque variable est ensuite jugée sur deux
chiffres : son lien au bien-être et sa corrélation avec le temps (pour éviter les proxys cachés).


In [ ]:
def build_features(frame: pd.DataFrame) -> pd.DataFrame:
    """Ratios de style, construits uniquement avec des champs d'usage."""
    f = pd.DataFrame(index=frame.index)
    total = frame[TIME_COLS].sum(axis=1).replace(0, np.nan)

    f["minutes_per_day"] = frame["daily_active_minutes_instagram"]
    f["log_minutes"] = np.log1p(frame["daily_active_minutes_instagram"])
    f["avg_session_min"] = frame["average_session_length_minutes"]
    f["sessions_per_day"] = frame["sessions_per_day"]
    f["feed_share"] = frame["time_on_feed_per_day"] / total
    f["reels_share"] = frame["time_on_reels_per_day"] / total
    f["msg_share"] = frame["time_on_messages_per_day"] / total
    f["explore_share"] = frame["time_on_explore_per_day"] / total
    f["style_score"] = (f["reels_share"] - f["feed_share"]).fillna(0)

    active_daily = (frame["likes_given_per_day"] + frame["comments_written_per_day"]
                    + frame["posts_created_per_week"] / 7 + frame["dms_sent_per_week"] / 7)
    f["interactions_per_min"] = active_daily / frame["daily_active_minutes_instagram"]

    passive = (frame["reels_watched_per_day"] + frame["stories_viewed_per_day"]
               + frame["ads_viewed_per_day"])
    f["passive_active_ratio"] = passive / (active_daily + 1)
    f["follower_ratio"] = np.log1p(frame["followers_count"]) - np.log1p(frame["following_count"])
    f["notification_response_rate"] = frame["notification_response_rate"]
    f["user_engagement_score"] = frame["user_engagement_score"]
    return f


features = build_features(df)
df["style_score"] = features["style_score"]

ratio_cols = ["feed_share", "reels_share", "msg_share", "explore_share",
              "interactions_per_min", "passive_active_ratio", "avg_session_min",
              "sessions_per_day", "notification_response_rate", "follower_ratio",
              "user_engagement_score"]

feature_audit = pd.DataFrame({
    "r stress": [features[c].corr(df["perceived_stress_score"]) for c in ratio_cols],
    "r bonheur": [features[c].corr(df["self_reported_happiness"]) for c in ratio_cols],
    "r sommeil": [features[c].corr(df["sleep_hours_per_night"]) for c in ratio_cols],
    "r log(minutes)": [features[c].corr(features["log_minutes"]) for c in ratio_cols],
}, index=ratio_cols)

def decision_for(row):
    if row.name in ["feed_share", "reels_share"]:
        return "Gardée — axe style"
    if abs(row["r log(minutes)"]) > 0.30:
        return "Écartée — confondue avec le temps"
    if abs(row["r stress"]) < 0.05:
        return "Écartée — lien nul"
    return "Descriptive"

feature_audit["Décision"] = feature_audit.apply(decision_for, axis=1)
display(feature_audit.round(3).sort_values("r stress", key=lambda s: s.abs(), ascending=False))

STYLE_THRESHOLD = float(features["style_score"].median())
print("Seuil de style (médiane de reels_share − feed_share) :", round(STYLE_THRESHOLD, 3))
print("Corrélation style_score / log(minutes) :", round(features["style_score"].corr(features["log_minutes"]), 4))
print("Corrélation style_score / stress :", round(features["style_score"].corr(df["perceived_stress_score"]), 4))


### Ce qui est gardé, ce qui est écarté

| Variable | Chiffre | Décision |
|---|---|---|
| `feed_share`, `reels_share` | `r(log minutes)` ≈ 0,03–0,04 ; `r(stress)` ≈ 0,02 | **Gardées** : indépendantes du temps, elles décrivent la surface de contenu. |
| `interactions_per_min` | `r(log minutes)` ≈ −0,84 | **Écartée** : malgré un lien au stress, c'est surtout un proxy du temps. |
| `passive_active_ratio` | `r(log minutes)` ≈ −0,46 | **Écartée** : confondue avec le temps. |
| `sessions_per_day` | `r(log minutes)` ≈ 0,64 | **Écartée** : la fréquence suit la quantité. |
| `avg_session_min` | `r(log minutes)` ≈ 0,36 | **Écartée** : corrélée au temps. |
| `notification_response_rate` | `r(stress)` ≈ 0,00 | **Écartée** : aucun lien. |
| `follower_ratio` | `r(stress)` ≈ 0,01 | **Écartée** : aucun lien. |
| `user_engagement_score` | `r(log minutes)` ≈ −0,85 | **Écartée** : score dérivé, redondant. |

**Conclusion B2.** Le seul axe de style à la fois **indépendant du temps** et **lisible produit**
est `style_score = reels_share − feed_share`. On le garde pour décrire les segments, en sachant
déjà qu'il ne déplacera pas le bien-être. Les ratios actif/passif, souvent cités dans le sujet,
sont des proxys du temps dans ces données : on les écarte de la segmentation.


## 12. B3 — Choisir la méthode : KMeans, prédictif ou hybride ?

Le sujet demande de comparer le clustering et la voie prédictive. On ajoute la voie **hybride**
(grille temps × style), qui est la seule à produire directement 4 à 7 segments nommés et
assignables.


In [ ]:
CLUSTER_COLS = ["log_minutes", "feed_share", "reels_share", "msg_share", "explore_share",
                "interactions_per_min", "passive_active_ratio", "avg_session_min",
                "sessions_per_day", "notification_response_rate"]

# --- 1) KMeans sur les comportements (diagnostic) ---
scaler_km = StandardScaler().fit(features[CLUSTER_COLS])
Z_all = scaler_km.transform(features[CLUSTER_COLS])
km_sample = features.sample(50_000, random_state=RANDOM_STATE)
Z_sample = scaler_km.transform(km_sample[CLUSTER_COLS])
km = KMeans(n_clusters=4, n_init=10, random_state=RANDOM_STATE).fit(Z_sample)
df["km_cluster"] = km.predict(Z_all)

sil_km = silhouette_score(Z_sample, km.labels_, sample_size=10_000, random_state=RANDOM_STATE)
ari_km = adjusted_rand_score(df["time_band"], df["km_cluster"])
eta_km = eta_squared(df, "km_cluster", "perceived_stress_score")

kmeans_profile = df.groupby("km_cluster").agg(
    n=("user_id", "size"),
    minutes=("daily_active_minutes_instagram", "mean"),
    reels_min=("time_on_reels_per_day", "mean"),
    feed_min=("time_on_feed_per_day", "mean"),
    stress=("perceived_stress_score", "mean"),
).sort_values("minutes")
display(kmeans_profile.round(2))
print(f"KMeans — silhouette : {sil_km:.3f} | ARI avec les bandes de temps : {ari_km:.3f} | η² stress : {eta_km:.3f}")

# --- 2) Voie prédictive : probabilité de stress élevé ---
X = features[CLUSTER_COLS]
y = df["high_stress"]
print("Cible prédictive : stress élevé (score ≥ 30/40), prévalence =", f"{y.mean():.1%}")
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=RANDOM_STATE, stratify=y
)

models = {
    "Temps seul (logit)": (["log_minutes"],
                           make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))),
    "Comportement complet (logit)": (CLUSTER_COLS,
                                     make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))),
    "Comportement complet (HGB)": (CLUSTER_COLS,
                                   HistGradientBoostingClassifier(
                                       max_iter=200, early_stopping=False, random_state=RANDOM_STATE)),
}
pred_rows = []
for name, (cols, model) in models.items():
    model.fit(X_train[cols], y_train)
    proba = model.predict_proba(X_test[cols])[:, 1]
    pred_rows.append({"Modèle": name, "AUC": roc_auc_score(y_test, proba),
                      "PR-AUC": average_precision_score(y_test, proba),
                      "Brier": brier_score_loss(y_test, proba)})
pred_results = pd.DataFrame(pred_rows).set_index("Modèle")
display(pred_results.round(3))

# --- 3) Hybride : 3 bandes de temps × 2 styles de contenu ---
df["style"] = np.where(features["style_score"] > STYLE_THRESHOLD, "Reels", "Feed")
df["segment_code"] = df["time_band"].astype(str) + " · " + df["style"]
SEGMENT_NAMES = {
    ("Faible", "Feed"): "Lecteurs occasionnels",
    ("Faible", "Reels"): "Scrolleurs occasionnels",
    ("Modéré", "Feed"): "Lecteurs réguliers",
    ("Modéré", "Reels"): "Scrolleurs réguliers",
    ("Intense", "Feed"): "Lecteurs intensifs",
    ("Intense", "Reels"): "Scrolleurs intensifs",
}
df["segment_name"] = [SEGMENT_NAMES[(band, style)]
                      for band, style in zip(df["time_band"].astype(str), df["style"])]

segment_profile = df.groupby("segment_name", observed=True).agg(
    n=("user_id", "size"),
    minutes=("daily_active_minutes_instagram", "mean"),
    sessions=("sessions_per_day", "mean"),
    reels_min=("time_on_reels_per_day", "mean"),
    feed_min=("time_on_feed_per_day", "mean"),
    stress=("perceived_stress_score", "mean"),
    bonheur=("self_reported_happiness", "mean"),
    sommeil=("sleep_hours_per_night", "mean"),
    stress_eleve=("high_stress", "mean"),
    mineurs=("is_minor", "mean"),
)
segment_profile["part"] = segment_profile["n"] / len(df)

eta_segment = {
    "stress": eta_squared(df, "segment_name", "perceived_stress_score"),
    "bonheur": eta_squared(df, "segment_name", "self_reported_happiness"),
    "sommeil": eta_squared(df, "segment_name", "sleep_hours_per_night"),
}
print("η² hybride → stress :", round(eta_segment["stress"], 3),
      "| bonheur :", round(eta_segment["bonheur"], 3),
      "| sommeil :", round(eta_segment["sommeil"], 3))
print("η² temps → stress :", round(eta_time["stress"], 3),
      "| Δ (hybride − temps) :", round(eta_segment["stress"] - eta_time["stress"], 4))


### Argumentation B3

- **KMeans** retrouve un gradient d'usage (un petit groupe à ~6 min, deux groupes moyens,
  un groupe intense), mais ses clusters ne coïncident pas avec les bandes de temps
  (ARI ≈ 0,07) et expliquent beaucoup moins bien le stress (η² ≈ 0,23 contre 0,63). Il est
  instable et difficile à nommer.
- **Prédictif** : le modèle complet atteint une AUC ≈ 0,922 contre 0,917 pour le temps seul.
  Il classe bien le risque, mais il produit une **probabilité**, pas un segment nommé ; il
  faut des labels de bien-être pour l'entraîner, et il ne dit pas quoi faire.
- **Hybride** : 3 bandes de temps × 2 styles = 6 segments, nommés, assignables, avec un axe
  produit clair. Il est aussi performant que le temps sur le stress (η² 0,633), mais il
  **n'ajoute rien** : le style est un descripteur, pas un levier de bien-être.

**Choix :** hybride pour la **segmentation produit** ; modèle prédictif en **complément** pour
prioriser les utilisateurs à risque ; bandes de temps pour toute lecture **bien-être**.


## 13. B4 — Construire et nommer les 6 segments

Les 6 segments sont les 6 cases de la grille. Les noms sont volontairement **produit**, pas
cliniques : un nom comme « Scrolleurs intensifs » décrit un usage, pas un état de santé.


In [ ]:
SEGMENT_ORDER = [
    "Lecteurs occasionnels", "Scrolleurs occasionnels",
    "Lecteurs réguliers", "Scrolleurs réguliers",
    "Lecteurs intensifs", "Scrolleurs intensifs",
]
segment_profile = segment_profile.reindex(SEGMENT_ORDER)
display(segment_profile.round(2))

product_actions = {
    "Lecteurs occasionnels": "Notifications douces, contenu feed ; aucun message santé.",
    "Scrolleurs occasionnels": "Découverte Reels ; aucun message santé.",
    "Lecteurs réguliers": "Feed long format ; mesurer l'engagement, pas le stress.",
    "Scrolleurs réguliers": "Reels ; mesurer l'engagement, pas le stress.",
    "Lecteurs intensifs": "Outils de contrôle du temps, opt-in ; pas de ciblage santé.",
    "Scrolleurs intensifs": "Outils de contrôle du temps, opt-in ; pas de ciblage santé.",
}

print("=== FICHES SEGMENTS ===")
for name in SEGMENT_ORDER:
    row = segment_profile.loc[name]
    print()
    print(f"[{name}]")
    print(f"  Taille        : {int(row['n']):,} users ({row['part']:.1%})")
    print(f"  Usage typique : {row['minutes']:.0f} min/jour, {row['sessions']:.1f} sessions, "
          f"{row['reels_min']:.0f} min Reels / {row['feed_min']:.0f} min Feed")
    print(f"  Bien-être     : stress {row['stress']:.1f}/40, bonheur {row['bonheur']:.1f}/10, "
          f"sommeil {row['sommeil']:.1f} h ; stress élevé {row['stress_eleve']:.1%}")
    print(f"  Mineurs       : {row['mineurs']:.1%} du segment")
    print(f"  Action produit: {product_actions[name]}")

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
x = np.arange(len(SEGMENT_ORDER))
axes[0].bar(x, segment_profile["stress"], color="#C44E52")
axes[0].set_xticks(x)
axes[0].set_xticklabels(SEGMENT_ORDER, rotation=35, ha="right", fontsize=8)
axes[0].set(title="Stress moyen par segment", ylabel="stress (0–40)")
axes[1].bar(x, segment_profile["bonheur"], color="#55A868")
axes[1].set_xticks(x)
axes[1].set_xticklabels(SEGMENT_ORDER, rotation=35, ha="right", fontsize=8)
axes[1].set(title="Bonheur moyen par segment", ylabel="bonheur (1–10)")
plt.tight_layout()
save_fig("03_06_segments.png")
plt.show()

style_table = df.groupby(["time_band", "style"], observed=True).agg(
    n=("user_id", "size"),
    stress=("perceived_stress_score", "mean"),
    bonheur=("self_reported_happiness", "mean"),
).round(2)
display(style_table)

eta_style = eta_squared(df, "style", "perceived_stress_score")
print("η² du style seul sur le stress :", round(eta_style, 4))
print("Différence de stress Feed vs Reels, par bande :")
print((df.groupby(["time_band", "style"], observed=True)["perceived_stress_score"]
       .mean().unstack().assign(difference=lambda d: d["Reels"] - d["Feed"]).round(3)).to_string())


**Conclusion B4.** Les 6 segments sont **équilibrés** (environ 1/6 chacun), **nommés** et
**interprétables**. Le stress moyen suit uniquement la bande de temps : 8,5 / 20,0 / 31,5.
À l'intérieur d'une bande, Feed et Reels ont des niveaux de bien-être **identiques** (écart
< 0,1 point). Le style de contenu est donc un axe **descriptif**, pas un axe de bien-être.


## 14. B5 — Affecter un nouvel utilisateur

La fonction n'utilise que des champs d'usage : minutes actives et temps par surface. Elle ne
regarde ni l'âge, ni le genre, ni un champ de santé. La démonstration se fait sur des profils
**fictifs**, jamais sur une ligne du fichier.


In [ ]:
def assign_segment(user: dict, time_bins=TIME_BINS, style_threshold=STYLE_THRESHOLD) -> str:
    """Renvoie le nom du segment pour un profil d'usage.

    Champs attendus : daily_active_minutes_instagram et les 4 time_on_*.
    Aucun champ de santé, d'âge ou de genre n'est utilisé.
    """
    total = sum(float(user.get(col, 0.0)) for col in TIME_COLS)
    if total <= 0:
        raise ValueError("Le profil doit avoir un temps d'usage strictement positif.")
    minutes = float(user["daily_active_minutes_instagram"])
    band = pd.cut([minutes], bins=time_bins, labels=TIME_LABELS,
                  include_lowest=True, right=True)[0]
    score = (float(user["time_on_reels_per_day"]) - float(user["time_on_feed_per_day"])) / total
    style = "Reels" if score > style_threshold else "Feed"
    return SEGMENT_NAMES[(str(band), style)]


fictitious_profiles = [
    {"Profil": "Fictif 1 — lecture légère",
     "daily_active_minutes_instagram": 45.0,
     "time_on_feed_per_day": 30.0, "time_on_reels_per_day": 8.0,
     "time_on_messages_per_day": 5.0, "time_on_explore_per_day": 2.0},
    {"Profil": "Fictif 2 — scroll léger",
     "daily_active_minutes_instagram": 45.0,
     "time_on_feed_per_day": 5.0, "time_on_reels_per_day": 35.0,
     "time_on_messages_per_day": 3.0, "time_on_explore_per_day": 2.0},
    {"Profil": "Fictif 3 — reels modéré",
     "daily_active_minutes_instagram": 200.0,
     "time_on_feed_per_day": 40.0, "time_on_reels_per_day": 120.0,
     "time_on_messages_per_day": 25.0, "time_on_explore_per_day": 15.0},
    {"Profil": "Fictif 4 — feed intense",
     "daily_active_minutes_instagram": 400.0,
     "time_on_feed_per_day": 250.0, "time_on_reels_per_day": 100.0,
     "time_on_messages_per_day": 30.0, "time_on_explore_per_day": 20.0},
]

assignment_rows = []
for profile in fictitious_profiles:
    total = sum(profile[col] for col in TIME_COLS)
    assignment_rows.append({
        "Profil": profile["Profil"],
        "Minutes": profile["daily_active_minutes_instagram"],
        "Part Reels": profile["time_on_reels_per_day"] / total,
        "Part Feed": profile["time_on_feed_per_day"] / total,
        "Segment": assign_segment(profile),
    })
assignments = pd.DataFrame(assignment_rows)
display(assignments.round(3))
display(assignments[["Profil", "Segment"]])

expected = ["Lecteurs occasionnels", "Scrolleurs occasionnels",
            "Scrolleurs réguliers", "Lecteurs intensifs"]
assert list(assignments["Segment"]) == expected, list(assignments["Segment"])
print("Affectation fictive validée : 4 profils → 4 segments attendus.")


**Conclusion B5.** La règle est **déterministe**, **expliquable** et **utilisable** sur un
nouvel utilisateur. Elle ne nécessite que 5 nombres d'usage. Elle ne produit pas de probabilité
et ne prétend pas prédire le stress ; c'est une **affectation de persona produit**.


## 15. C2 — Coder les tests des critères

Une fonction par critère, un chiffre et un verdict. Les seuils sont ceux de la section C1 :
ils ne sont pas modifiés après avoir vu les résultats.


In [ ]:
CRIT_DISTINCT_MIN = 0.45
CRIT_STABLE_MIN = 0.90
CRIT_WELLBEING_MIN = 0.30
CRIT_INCREMENTAL_MIN = 0.01
CRIT_MIN_PART = 0.05
CRIT_SEG_MIN, CRIT_SEG_MAX = 4, 7

df["log_minutes"] = features["log_minutes"]


def crit_distinct() -> dict:
    """Distance minimale entre centroïdes des 6 segments sur les 2 axes standardisés."""
    X = StandardScaler().fit_transform(df[["log_minutes", "style_score"]])
    cent = pd.DataFrame(X, columns=["temps", "style"]).groupby(df["segment_name"]).mean()
    distances = [float(np.linalg.norm(cent.loc[a] - cent.loc[b]))
                 for a, b in combinations(cent.index, 2)]
    value = min(distances)
    return {"Critère": "1. Distincts", "Mesure": "distance min. entre centroïdes",
            "Seuil": f"≥ {CRIT_DISTINCT_MIN:.2f}", "Résultat": f"{value:.3f}",
            "Verdict": "Respecté" if value >= CRIT_DISTINCT_MIN else "Non respecté"}


def thresholds_from(frame: pd.DataFrame) -> tuple:
    q1_, q2_ = frame["daily_active_minutes_instagram"].quantile([1/3, 2/3])
    total = frame[TIME_COLS].sum(axis=1)
    st = (frame["time_on_reels_per_day"] / total
          - frame["time_on_feed_per_day"] / total).median()
    return float(q1_), float(q2_), float(st)


def assign_vector(frame: pd.DataFrame, thresholds: tuple) -> np.ndarray:
    q1_, q2_, st = thresholds
    bands = pd.cut(frame["daily_active_minutes_instagram"], [0, q1_, q2_, np.inf],
                   labels=TIME_LABELS, include_lowest=True, right=True).astype(str)
    total = frame[TIME_COLS].sum(axis=1)
    score = frame["time_on_reels_per_day"] / total - frame["time_on_feed_per_day"] / total
    styles = np.where(score > st, "Reels", "Feed")
    return np.array([SEGMENT_NAMES[(band, style)] for band, style in zip(bands, styles)])


def crit_stable() -> dict:
    """Kappa entre deux moitiés tirées au hasard : les seuils doivent peu bouger."""
    rng = np.random.default_rng(RANDOM_STATE)
    idx = rng.permutation(len(df))
    h1 = df.iloc[idx[:len(df) // 2]]
    h2 = df.iloc[idx[len(df) // 2:]]
    labels_1 = assign_vector(df, thresholds_from(h1))
    labels_2 = assign_vector(df, thresholds_from(h2))
    value = cohen_kappa_score(labels_1, labels_2)
    return {"Critère": "2. Stables", "Mesure": "kappa split-half",
            "Seuil": f"≥ {CRIT_STABLE_MIN:.2f}", "Résultat": f"{value:.3f}",
            "Verdict": "Respecté" if value >= CRIT_STABLE_MIN else "Non respecté"}


def crit_wellbeing() -> dict:
    value = eta_squared(df, "segment_name", "perceived_stress_score")
    return {"Critère": "3. Liés au bien-être", "Mesure": "η² stress",
            "Seuil": f"≥ {CRIT_WELLBEING_MIN:.2f}", "Résultat": f"{value:.3f}",
            "Verdict": "Respecté" if value >= CRIT_WELLBEING_MIN else "Non respecté"}


def crit_incremental() -> dict:
    value = eta_squared(df, "segment_name", "perceived_stress_score") - eta_time["stress"]
    return {"Critère": "4. Plus informatifs que le temps", "Mesure": "Δη² vs bandes de temps",
            "Seuil": f"≥ {CRIT_INCREMENTAL_MIN:.2f}", "Résultat": f"{value:+.3f}",
            "Verdict": "Respecté" if value >= CRIT_INCREMENTAL_MIN else "Non respecté"}


def crit_size() -> dict:
    counts = df["segment_name"].value_counts()
    n_segments = len(counts)
    min_part = counts.min() / len(df)
    ok = (CRIT_SEG_MIN <= n_segments <= CRIT_SEG_MAX) and (min_part >= CRIT_MIN_PART)
    return {"Critère": "5. Nombre et taille", "Mesure": "nb segments ; part du plus petit",
            "Seuil": "4–7 ; ≥ 5 %", "Résultat": f"{n_segments} ; {min_part:.1%}",
            "Verdict": "Respecté" if ok else "Non respecté"}


def crit_assignable() -> dict:
    expected = ["Lecteurs occasionnels", "Scrolleurs occasionnels",
                "Scrolleurs réguliers", "Lecteurs intensifs"]
    obtained = [assign_segment(p) for p in fictitious_profiles]
    n_ok = sum(a == b for a, b in zip(obtained, expected))
    return {"Critère": "6. Assignable", "Mesure": "profils fictifs corrects",
            "Seuil": "100 %", "Résultat": f"{n_ok}/{len(expected)}",
            "Verdict": "Respecté" if n_ok == len(expected) else "Non respecté"}


criteria_results = pd.DataFrame([
    crit_distinct(), crit_stable(), crit_wellbeing(),
    crit_incremental(), crit_size(), crit_assignable(),
])
display(criteria_results)
print("Critères respectés :", int((criteria_results["Verdict"] == "Respecté").sum()),
      "/", len(criteria_results))


## 16. C3 — Évaluer les segments de B

### Tableau de synthèse

Le tableau ci-dessus est le « jury avant le jury ». Cinq critères passent, **le critère 4 échoue** :

| Critère | Résultat | Lecture |
|---|---|---|
| 1. Distincts | distance min. ≈ 0,50 | Les 6 cases sont bien séparées. |
| 2. Stables | kappa ≈ 0,99 | Les seuils ne dépendent pas de l'échantillon. |
| 3. Liés au bien-être | η² ≈ 0,63 | Aussi fort que le temps seul. |
| 4. Plus informatifs que le temps | Δη² ≈ 0,00 | **Échec** : le style n'ajoute rien. |
| 5. Nombre et taille | 6 segments ; ≈ 16,6 % chacun | Utilisable. |
| 6. Assignable | 4/4 profils fictifs | La règle fonctionne. |

### Ce qu'on renvoie au binôme B

- Le **nom et la taille** des segments tiennent : 6 segments, aucun segment minuscule.
- La **stabilité** tient : kappa ≈ 0,99.
- Le **lien au bien-être** tient, mais il vient **entièrement de l'intensité**.
- L'axe **style** ne tient pas comme axe de bien-être : il ne faut pas vendre les 6 segments
  comme « plus prédictifs » que les 3 bandes de temps.
- Décision : garder les 6 segments comme **segmentation produit** (contenu, activation), et
  utiliser les 3 bandes de temps pour toute lecture de **risque bien-être**. Le modèle prédictif
  de la route 02 reste le complément pour prioriser, pas pour cibler.

### Comparaison finale

| Route | Mesure | Résultat |
|---|---|---|
| Temps seul (3 bandes) | η² stress | ≈ 0,633 |
| KMeans (4 clusters) | η² stress | ≈ 0,232 |
| Prédictif (HGB) | AUC stress élevé | ≈ 0,922 (temps seul ≈ 0,917) |
| Hybride (6 segments) | η² stress | ≈ 0,633 (Δ ≈ 0,00) |


## 17. C4 — Limites

**1. Corrélation n'est pas causalité.** Le temps d'usage et le stress sont fortement associés
(r ≈ 0,83 ; η² ≈ 0,63), mais rien ne dit que c'est l'usage qui augmente le stress. Une personne
stressée peut aussi utiliser davantage Instagram, et des facteurs tiers (sommeil, travail,
isolement) peuvent expliquer les deux. Nos segments sont descriptifs : ils ne prouvent pas qu'une
réduction du temps ferait baisser le stress.

**2. Un panel de volontaires ne représente pas tous les utilisateurs.** Les 200 000 lignes sont
un extrait synthétique et opt-in ; 9,5 % sont des mineurs, dont 2 401 au Royaume-Uni ou en
Allemagne. Les proportions, les moyennes et les segments ne sont pas des estimations de la
population Instagram réelle. Un déploiement devrait refaire l'audit sur un échantillon
représentatif et documenter le taux de réponse.

**3. Des données synthétiques ne disent pas tout des vrais utilisateurs.** Le sommeil ne bouge
presque pas (r ≈ 0,00), le style de contenu n'ajoute aucune variance, et 8 828 mineurs ont encore
une éducation invraisemblable (Bachelor's/Master's) malgré le nettoyage. Le plancher à 5,0 min/jour
et les dates de création corrigées sont des artefacts à documenter. Ces résultats valent pour ce
générateur, pas comme lois universelles.


In [ ]:
limits_facts = pd.DataFrame([
    {"Fait": "r(temps, stress)", "Valeur": df["daily_active_minutes_instagram"].corr(df["perceived_stress_score"])},
    {"Fait": "r(temps, bonheur)", "Valeur": df["daily_active_minutes_instagram"].corr(df["self_reported_happiness"])},
    {"Fait": "r(temps, sommeil)", "Valeur": df["daily_active_minutes_instagram"].corr(df["sleep_hours_per_night"])},
    {"Fait": "η² temps → stress", "Valeur": eta_time["stress"]},
    {"Fait": "η² style → stress", "Valeur": eta_style},
    {"Fait": "Mineurs", "Valeur": int(df["is_minor"].sum())},
    {"Fait": "Mineurs UE/UK", "Valeur": int(((df["is_minor"] == 1) & df["country"].isin(["Germany", "United Kingdom"])).sum())},
    {"Fait": "Éducation mineurs invraisemblable", "Valeur": int(df["education_minor_implausible"].sum())},
    {"Fait": "Utilisateurs au plancher 5 min", "Valeur": int(df["usage_floor_flag"].sum())},
    {"Fait": "Dates de création corrigées", "Valeur": int((df["account_creation_year"].values != raw_a["account_creation_year"].values).sum())},
])
def _fmt_limit(v):
    fv = float(v)
    return f"{int(fv):,}" if fv.is_integer() else f"{fv:,.3f}"

limits_facts["Valeur"] = [_fmt_limit(v) for v in limits_facts["Valeur"]]
display(limits_facts)


## 18. C5 — Position RGPD : UE et Royaume-Uni

> **Avertissement.** Ceci est une analyse de conformité pour un exercice pédagogique, pas un avis
> juridique. Les articles sont cités par leur numéro ; les liens renvoient aux textes officiels.

### Question 1 — Sur quelles variables peut-on construire des segments de ciblage ?

**Possible, avec conditions :** des variables d'**usage** (temps, sessions, surfaces consultées)
et des variables de **profil non sensibles** (pays, langue, type d'appareil), pour une finalité
déterminée, explicite et légitime, avec base légale (consentement ou intérêt légitime), minimisation
et information des personnes.

**Interdit en l'état :** utiliser le stress, le bonheur, le sommeil, l'IMC, la tension, la santé
ou un mode de vie comme variables de ciblage. Ce sont des données de santé au sens de l'article 9
du RGPD ; elles exigent une base spécifique (consentement explicite, etc.) que l'étude ne fournit
pas. Les variables de profil comme `income_level`, `relationship_status` ou `has_children` ne sont
pas nécessaires ici et augmentent le risque de discrimination : on ne les utilise pas.

### Question 2 — Placer quelqu'un dans un segment révèle-t-il sa santé ?

**Oui, dans nos données.** Les 6 segments sont construits uniquement sur l'usage, mais leur
appartenance prédit fortement le stress : stress moyen ≈ 8,5 / 20,0 / 31,5 selon la bande de temps,
η² ≈ 0,63. Un segment est donc un **proxy d'état de santé**, même sans champ santé en entrée.
Cela déclenche :

- l'article 9 (données de santé) : base spécifique et garanties renforcées ;
- l'article 22 si la décision produit un effet significatif (publicité, accès, prix) : droit
  d'information, d'opposition et d'explication ;
- l'article 35 : analyse d'impact (DPIA) avant le traitement ;
- les principes de minimisation et de limitation de finalité (article 5).

**Conséquence :** les segments peuvent servir à de l'**analytique agrégée** ou à des **outils
opt-in de contrôle du temps**, jamais à du ciblage publicitaire santé, à une inférence de stress
individuelle, ni à une décision automatisée défavorable.

### Question 3 — Que change la présence de mineurs ?

**18 913 mineurs (9,5 %)**, dont **2 401 au Royaume-Uni ou en Allemagne**. Le profilage des
mineurs est très encadré :

- **UE** : article 8 du RGPD — consentement parental pour les services de la société de
  l'information lorsque le mineur a moins de 16 ans (la France a fixé l'âge à 15 ans) ; les
  États membres peuvent abaisser l'âge, pas en dessous de 13 ans.
- **Royaume-Uni** : UK GDPR + Data Protection Act 2018, et **Age Appropriate Design Code** de
  l'ICO (15 standards) : intérêts supérieurs de l'enfant, minimisation, pas de profilage
  préjudiciable, paramètres protecteurs par défaut.
- **Dans notre cas** : les mineurs ne doivent pas être ciblés par les segments, encore moins par
  un message santé. Si les segments sont utilisés, il faut **exclure les mineurs** ou agrégéger
  sans jamais prendre de décision individuelle. Une DPIA spécifique enfants est nécessaire.

### Tableau de position

| Question | UE | Royaume-Uni | Conditions |
|---|---|---|---|
| Variables de ciblage | Usage + profil non sensible | Idem | Base légale, finalité, minimisation, information |
| Segment = santé ? | Oui, proxy de stress | Oui, donnée de santé | Art. 9, DPIA, pas de ciblage santé |
| Mineurs | Art. 8, 13–16 ans | AADC, 15 standards | Exclusion du ciblage, DPIA enfants, consentement parental |

### Sources officielles

- RGPD consolidé : <https://eur-lex.europa.eu/eli/reg/2016/679/oj>
- Article 9 (catégories particulières) : <https://eur-lex.europa.eu/legal-content/FR/TXT/?uri=CELEX:32016R0679>
- Article 8 (enfants) : même texte, article 8.
- Article 22 (décision automatisée) et article 35 (DPIA) : même texte.
- CNIL — RGPD : <https://www.cnil.fr/fr/reglement-europeen-protection-donnees>
- ICO — UK GDPR : <https://ico.org.uk/for-organisations/uk-gdpr-guidance-and-resources/>
- ICO - Children's code : <https://ico.org.uk/for-organisations/uk-gdpr-guidance-and-resources/childrens-information/childrens-code-guidance-and-resources/>

*Liens vérifiés le 22/10/2026 (EUR-Lex : 202, ICO : 200).*


## 19. C6 — Trame des slides (12 slides, 15 minutes)

Un seul fil narratif : **le temps explique le bien-être ; le style décrit l'usage ; on ne cible
ni la santé ni les mineurs.**

| # | Titre | Message | Visuel | Pilote |
|---|---|---|---|---|
| 1 | Titre | 200 000 utilisateurs, 58 colonnes, une question : peut-on segmenter utilement ? | Titre + chiffres clés | C |
| 2 | Les données | Deux extraits, 6 champs divergents, un choix documenté | Tableau A1/A2 | A |
| 3 | Les 3 tiroirs | Usage / profil / bien-être : ce qui a le droit de servir | Tableau T1 | A |
| 4 | Ce que disent les données | Temps ↔ stress : r ≈ 0,83 ; sommeil : r ≈ 0 | Heatmap + déciles | A |
| 5 | La référence temps | 3 bandes : 8,5 / 20,0 / 31,5 de stress | Tableau B1 | B |
| 6 | Les ratios | Reels/Feed indépendant du temps ; actif/passif = proxy du temps | Tableau B2 | B |
| 7 | Trois méthodes | KMeans, prédictif, hybride : ce que chacune apprend | Tableau B3 | B |
| 8 | Les 6 segments | Noms, tailles, usages, bien-être | Graphique B4 | B |
| 9 | Affectation | Un profil fictif → un segment, en 5 nombres | Démo B5 | B |
| 10 | Les critères | 5/6 passent ; le style n'ajoute rien | Tableau C3 | C |
| 11 | Limites | Corrélation ≠ causalité ; panel ; données synthétiques | 3 chiffres | C |
| 12 | RGPD & décision | Segments produit, pas de ciblage santé/mineurs | Tableau C5 | C |

**Répétition :** 12 slides × ~1 min 10, puis 3 min de questions. Chaque binôme apporte **un
visuel** ; C vérifie que le fil reste unique (pas trois présentations collées).


## 20. Conclusion — ce qu'on garde, ce qu'on refuse

**Ce qu'on garde.** Une segmentation **produit** en 6 personas, stable, équilibrée et assignable :
`Lecteurs/Scrolleurs` × `occasionnels/réguliers/intensifs`. Elle est utile pour décider d'un
format de contenu ou d'un outil d'activation.

**Ce qu'on refuse.** Dire que ces 6 segments sont « meilleurs que le temps » pour le bien-être :
le critère 4 échoue (Δη² ≈ 0,00). Dire qu'ils permettent de cibler la santé : c'est interdit sans
base article 9 et dangereux. Dire qu'ils s'appliquent aux mineurs : non, 18 913 mineurs sont dans
le panel et doivent être exclus du ciblage.

**Décision opérationnelle :**

1. **Bien-être / risque** → 3 bandes de temps, simple et fort (η² ≈ 0,63).
2. **Produit / contenu** → 6 segments, lisibles et stables.
3. **Priorisation individuelle** → modèle prédictif de la route 02, avec calibration et audit
   d'équité, jamais comme décision automatique.
4. **Gouvernance** → DPIA, exclusion des mineurs, pas de champs santé en entrée, pas de ciblage
   santé, agrégats uniquement.


In [ ]:
auc_time = pred_results.loc["Temps seul (logit)", "AUC"]
auc_full = pred_results.loc["Comportement complet (HGB)", "AUC"]

print("=== SYNTHÈSE FINALE (calculée à l'exécution) ===")
print(f"Segments                         : {len(segment_profile)}")
print(f"Plus petit segment               : {segment_profile['part'].min():.1%}")
print(f"η² stress — temps seul           : {eta_time['stress']:.3f}")
print(f"η² stress — 6 segments           : {eta_segment['stress']:.3f}")
print(f"Δ (6 segments − temps)           : {eta_segment['stress'] - eta_time['stress']:+.4f}")
print(f"η² stress — style seul           : {eta_style:.4f}")
print(f"AUC stress élevé — temps seul    : {auc_time:.3f}")
print(f"AUC stress élevé — modèle complet: {auc_full:.3f}")
print(f"Kappa de stabilité               : {criteria_results.loc[1, 'Résultat']}")
print(f"Critères respectés               : {(criteria_results['Verdict'] == 'Respecté').sum()}/{len(criteria_results)}")
print()
print("Message pour le jury : le temps porte le signal de bien-être ; le style de contenu")
print("est un axe produit utile mais nul sur le bien-être. On livre 6 segments pour le produit,")
print("3 bandes pour le risque, et aucune utilisation santé/mineur.")


## 21. Annexe — Checklist des tâches

| Tâche | Statut | Preuve dans ce notebook |
|---|---|---|
| T0 — dépôt Git, rôles, README | Procédure posée | Section 0 ; tableau des binômes |
| T1 — classer les 58 colonnes | Fait | Section 3 ; 58 lignes, 3 tiroirs |
| A1 — comparer les deux fichiers | Fait | Section 4 ; 6 colonnes, comptes |
| A2 — enquêter sur chaque différence | Fait | Section 5 ; preuves + décision |
| A3 — jeu de données final | Fait | Section 6 ; `charger_donnees()` |
| A4 — explorer usage et bien-être | Fait | Section 7 ; 4 graphiques commentés |
| A5 — reproductibilité | Fait | Section 8 ; double exécution + hash |
| B1 — référence temps | Fait | Section 10 ; tableau + η² |
| B2 — nouvelles variables | Fait | Section 11 ; ratios + décisions |
| B3 — choix de méthode | Fait | Section 12 ; KMeans vs prédictif vs hybride |
| B4 — segments nommés | Fait | Section 13 ; 6 fiches |
| B5 — affectation | Fait | Section 14 ; 4 profils fictifs |
| C1 — critères | Fait | Section 9 ; 6 critères avant résultats |
| C2 — tests | Fait | Section 15 ; 6 fonctions |
| C3 — évaluation | Fait | Section 16 ; tableau + renvoi à B |
| C4 — limites | Fait | Section 17 ; 3 paragraphes chiffrés |
| C5 — RGPD | Fait | Section 18 ; UE / UK / mineurs |
| C6 — slides | Fait | Section 19 ; 12 slides |

**Reste à faire dans le dépôt (hors notebook) :** commits nominatifs, README avec le tableau des
binômes, captures des 12 slides, répétition chronométrée.
